# SentinelFusion 

An experiment to test bottleneck transformer to mix image and weather features.<br><br/>
Architecture : 
- Images are fed to a pretrained ViT model to get a `(14 14)` tensor of embedded features
- Weather data is fed to an encoder that encodes them to a `(14, 14)` tensor of embedded features
- The two outputs are then concatenated to a  `(28, 14)` tensor that is fed to a final bottleneck encoder
- The encoder outputs probabilities of having or not the mildiou in the field

## Imports and settings

Here are some imports and settings, with an initialization function that sets the seed for everything in our workspace from `torch` to `numpy`

In [ ]:
import os
import random
import cv2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.utils.data as torch_data
import torchsummary as tsum
import transformers
from PIL import Image
from torchvision import transforms

from dataset.dataset import NDVIDiseaseDataset
from manager import *
from sentinel_model import SentinelFusion
from utils import *

# Seaborn
sns.set()
# GPU device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Seed setting 
seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 
torch.use_deterministic_algorithms(True, warn_only=True)
os.environ['CUBLAS_WORKSPACE_CONFIG']=':16:8'
os.environ['PYTHONHASHSEED'] = str(seed)

print(f"[Workspace initialized !]")

## Network configuration

The network uses a Python dictionary as configuration, with a lot of attributes as the input dimensions, model dimensions, number of layers, etc.
The configuration is split into 4 sub-configurations.
- The `global_config` part defines the device on which to load everything, the number of previous days the model uses, and the global dropout
- The `wam_config` part (`WeatherAttentionModel`) defines all the parameters of the WeatherAttentionModel
- The `fusion_config` part (`FusionBottleneck`) defines all the parameters of the fusion bottleneck encoder
- The `vit_config` part (`ViT`) defines all the parameters of the ViT Transformer used to generate the image embedded features

In [ ]:
MODEL_CONFIG = {
    "global_config":
    {
        "device": device,
        "dropout": 0.3,
    },
    "vit_config":{
        "n_channels": 9,
        "patch_size": 16,
        "emb_size": 768,
        "img_size": 224,
        "n_layers":12,
        "n_attn_heads": 8,
        "fc_exp":4,
        "dropout": 0.3,
    },
    "wam_config": 
    {        
        "input_dim": 12,
        "model_dim": 64,
        "n_layers": 12,
        "n_attn_heads": 8,
        "output_dim": 196,
        "ff_dim": 128,
        "reshape": (14, 14),
    },
    "fusion_config":
    {
        "input_dim": 14,
        "model_dim": 64, 
        "n_layers": 12,
        "n_attn_heads": 8,
        "output_dim": 2,
        "ff_dim": 128,
    }
}


## Loading data 

We are going to load our data by using the `NDVIDiseaseDataset` defined in `./dataset/dataset.py` that I created beforehand which returns for each index a dictionary looking like : 
```json
{
    "image": torch.Size(img_size), 
    "weather": torch.Size(1, 12), 
    "target": [0, 1, 0]
}
```

In [ ]:
train_dataset = NDVIDiseaseDataset(root_dir='./dataset', train=True, vi_discard=[])
test_dataset = NDVIDiseaseDataset(root_dir='./dataset', train=False, vi_discard=[])
# Validation set is 15% of the training set
train_size = int(len(train_dataset) * 0.75)
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = torch_data.DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=8)
test_loader = torch_data.DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=8)
val_loader = torch_data.DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=8)

print("[Loaded dataset with {} images (train_set={}, test_set={})]".format(len(train_dataset)+len(test_dataset), len(train_dataset), len(test_dataset)))
print(f"Image Vector Shape: {train_dataset[0]['image'].shape}\nWeather Vector Shape: {train_dataset[0]['weather'].shape}")

## Loading model

We can now load our `SentinelFusion` model using the configuration. The model is then passed to the `ModelManager`, a class created to handle the training, testing, and a lot of other stuff about the SentinelFusion model.

In [ ]:
# Creating the model by passing the config 
model = SentinelFusion(MODEL_CONFIG)
model.vit_model.load_pretrained_weights()
# Print a small summary of the model parameters
model.summary()

# Creating manager and loading pretrained weights of ViTT
manager = ModelManager(
    s_model=model,
    data_loaders={
        "train": train_loader,
        "test": test_loader,
        "val": val_loader
    }
)

## Training the model

*(You can skip this section if you want to test the model, loading is available)*

We can train the model with the `train()` function of the `SentinelFusionManager`.
The optimizer chosen is `Adam` and we use a special learning rate scheduler (a cosine warmup scheduler) which helps a lot the transformers to perform better in training.
The loss is cross-entropy since this is a classification task.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)
lr_sched = transformers.optimization.get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=600)
criterion = torch.nn.CrossEntropyLoss()

# Training the model (commented because we will load the model for testing)
manager.train(criterion, optimizer, lr_sched, n_epochs=600, checkpoint=False, early_stop=True)
plt.figure(figsize=(15,8))
plt.plot(list([manager.hist['train_loss'][x] for x in range(len(manager.hist['train_loss'])) if x%20==0]))
plt.plot(list([manager.hist['val_loss'][x] for x in range(len(manager.hist['val_loss'])) if x%20==0]))
plt.legend(['train_loss', 'val_loss'])
plt.title('Training losses')

## Testing and evaluating the model

In this section we will test the model and its performances. First, we can load the model stored in `checkpoints` to get a fully functional model.

In [ ]:
# Testing the mode
stats = manager.test(criterion)
print(stats)

## Attention vizualisation

In this section we generate the attention map and outputs for a random sample of the test set.

In [ ]:
with torch.no_grad():
    # Generating seed 
    img_seed = 500
    # Titles 
    titles = ['NDVI Rainbow', 'NDCI', 'MOISTURE', 'Attention', 'Attention Interpolated']

    # Getting image name, prediction
    img_name = test_dataset._get_img_name(img_seed)
    img, weather = test_dataset[img_seed]['image'].unsqueeze(0).to(device), test_dataset[img_seed]['weather'].unsqueeze(0).to(device)
    model_output = model(img, weather)
    # Getting attention
    attn_list = manager.model.vit_model.get_attn(img).detach().cpu().numpy().squeeze()

    # Plotting
    fig, ax = plt.subplots(1,5, figsize=(30,8))
    ax[0].imshow(test_dataset[img_seed]['image'][:3,:,:].permute(1,2,0).clip(0,1))
    ax[1].imshow(test_dataset[img_seed]['image'][3:6,:,:].permute(1,2,0).clip(0,1))
    ax[2].imshow(test_dataset[img_seed]['image'][6:9,:,:].permute(1,2,0).clip(0,1))
    ax[3].imshow(attn_list.mean(axis=0).mean(axis=0)[1:].reshape(14, 14).clip(0,1))
    ax[4].imshow(cv2.resize(attn_list.sum(axis=0).mean(axis=0)[1:].reshape(14, 14).clip(0,1), (224, 224), interpolation=cv2.INTER_CUBIC))

    for i in range(len(ax)):
        ax[i].set_title(titles[i])
        ax[i].axis('off')

    fig.suptitle(f"Predictions for {img_name}", fontsize=20)
    fig.text(0.5, 0.90, f"Model output = {model_output.argmax().item()} ({'mildiou' if model_output.argmax().item()==0 else 'pas de mildiou'})", fontsize=15, ha='center')
    fig.text(0.5, 0.85, f"True label = {test_dataset[img_seed]['target'].argmax().item()} ({'mildiou' if test_dataset[img_seed]['target'].argmax().item()==0 else 'pas de mildiou'})", fontsize=15, ha='center')